In [9]:
# =============================================================================
# PDB → PDBQT CONVERSION WITH HYDROGENS (Colab)
# =============================================================================
# This script:
#   1. Installs OpenBabel and required packages.
#   2. Provides a file upload button for your PDB files (or a ZIP archive).
#   3. Converts each PDB to PDBQT with hydrogens and Gasteiger charges.
#   4. Cleans the PDBQT files (strips headers).
#   5. Zips all PDBQT files and downloads them.
# =============================================================================

# -----------------------------------------------------------------------------
# 1. Install OpenBabel and dependencies
# -----------------------------------------------------------------------------
!apt-get update -qq > /dev/null 2>&1
!apt-get install -y openbabel > /dev/null 2>&1
!pip install -q tqdm

import os
import glob
import zipfile
import subprocess
from tqdm import tqdm
from google.colab import files

# -----------------------------------------------------------------------------
# 2. Create directories
# -----------------------------------------------------------------------------
UPLOAD_DIR = "/content/uploaded_pdbs"
OUTPUT_DIR = "/content/pdbqt_output"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"📁 Upload directory: {UPLOAD_DIR}")
print(f"📁 Output directory: {OUTPUT_DIR}")

# -----------------------------------------------------------------------------
# 3. Upload your PDB files (or a ZIP archive)
# -----------------------------------------------------------------------------
print("\n📤 Please upload your clean PDB files (or a ZIP containing them).")
uploaded = files.upload()

# Save uploaded files
for filename, content in uploaded.items():
    filepath = os.path.join(UPLOAD_DIR, filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"✅ Saved: {filename}")

# Extract ZIP archives if any
zip_files = [f for f in os.listdir(UPLOAD_DIR) if f.endswith('.zip')]
for zip_name in zip_files:
    zip_path = os.path.join(UPLOAD_DIR, zip_name)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(UPLOAD_DIR)
    print(f"📦 Extracted: {zip_name}")
    os.remove(zip_path)  # remove the zip after extraction

# Find all PDB files
pdb_files = glob.glob(os.path.join(UPLOAD_DIR, '*.pdb'))
if not pdb_files:
    raise SystemExit("❌ No PDB files found. Please upload .pdb files or a ZIP archive.")

print(f"\n🔍 Found {len(pdb_files)} PDB files.")

# -----------------------------------------------------------------------------
# 4. Conversion function
# -----------------------------------------------------------------------------
def convert_pdb_to_pdbqt(pdb_path, pdbqt_path):
    """
    Convert a PDB file to PDBQT using OpenBabel.
    Adds hydrogens (-h) and Gasteiger charges at pH 7.4 (-p 7.4).
    Then strips headers to keep only ATOM/HETATM lines.
    """
    cmd = [
        'obabel',
        pdb_path,
        '-O', pdbqt_path,
        '-h',          # add hydrogens
        '-p', '7.4'    # assign charges at physiological pH
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"⚠️ Error converting {os.path.basename(pdb_path)}: {result.stderr[:200]}")
        return False

    # Clean the PDBQT: keep only ATOM and HETATM lines
    with open(pdbqt_path, 'r') as f:
        lines = f.readlines()
    clean_lines = [line for line in lines if line.startswith(('ATOM', 'HETATM'))]
    with open(pdbqt_path, 'w') as f:
        f.writelines(clean_lines)

    return True

# -----------------------------------------------------------------------------
# 5. Batch conversion
# -----------------------------------------------------------------------------
print("\n🔄 Converting PDB to PDBQT (with hydrogens)...")
success_count = 0

for pdb_path in tqdm(pdb_files, desc="Converting"):
    base = os.path.splitext(os.path.basename(pdb_path))[0]
    pdbqt_path = os.path.join(OUTPUT_DIR, base + '.pdbqt')

    # Skip if already converted
    if os.path.exists(pdbqt_path):
        print(f"⏭️  Skipping {base} (already exists)")
        success_count += 1
        continue

    if convert_pdb_to_pdbqt(pdb_path, pdbqt_path):
        success_count += 1

print(f"\n✅ Successfully converted {success_count} out of {len(pdb_files)} files.")

# -----------------------------------------------------------------------------
# 6. Zip and download
# -----------------------------------------------------------------------------
zip_path = "/content/pdbqt_files.zip"
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for pdbqt in glob.glob(os.path.join(OUTPUT_DIR, '*.pdbqt')):
        zipf.write(pdbqt, os.path.basename(pdbqt))

print(f"\n📦 Zipped {success_count} PDBQT files to: {zip_path}")
files.download(zip_path)

print("\n✅ All done! Your PDBQT files are ready for Vina docking.")

📁 Upload directory: /content/uploaded_pdbs
📁 Output directory: /content/pdbqt_output

📤 Please upload your clean PDB files (or a ZIP containing them).


Saving Wild-Type TP53 Sequence UniProt.pdb to Wild-Type TP53 Sequence UniProt.pdb
✅ Saved: Wild-Type TP53 Sequence UniProt.pdb

🔍 Found 1 PDB files.

🔄 Converting PDB to PDBQT (with hydrogens)...


Converting: 100%|██████████| 1/1 [00:40<00:00, 40.14s/it]


✅ Successfully converted 1 out of 1 files.

📦 Zipped 1 PDBQT files to: /content/pdbqt_files.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All done! Your PDBQT files are ready for Vina docking.
